# Repair corpus tree — FULL (crash-safe, resumable)

Repairs `corpus_tree.json` by finding tables (incl. checklist tables) skipped during the original build and inserting each as its **own `section` node** (level 2) containing one table `chunk`, placed among the sibling sections in the correct position. Node format: UUID4 `node_id`, `name` `<file> - <label> - table 0`, full raw table in `content`, metadata `source_file, section, unit_index, file_type, kind:'table', page`. **Crash-safe:** writes `corpus_tree_fixed.json` after every file and records processed files, so if it stops you can just re-run this notebook and it resumes where it left off. Idempotent — re-running never double-inserts.

In [1]:
import sys, subprocess
for pkg in ['pdfplumber','ollama','tqdm']:
    try: __import__(pkg)
    except Exception: subprocess.run([sys.executable,'-m','pip','install','-q',pkg])
print('deps ready')

deps ready


In [2]:
from pathlib import Path
PROJECT_DIR = Path('.').resolve()
TREE_PATH   = PROJECT_DIR / 'tree_cache' / 'corpus_tree.json'
FIXED_PATH  = PROJECT_DIR / 'tree_cache' / 'corpus_tree_fixed.json'   # written after every file
PROGRESS_PATH = PROJECT_DIR / 'tree_cache' / 'repair_processed.json'  # which files are done (for resume)
FOLDERS_DIR = PROJECT_DIR / 'folders'
OLLAMA_URL  = 'http://localhost:11528'
AGENT_MODEL = 'gpt-oss:120b'
EMBED_MODEL = 'nomic-embed-text'
print('source tree :', TREE_PATH)
print('output tree :', FIXED_PATH)

source tree : /Users/asharma/Desktop/Project Algorithm/tree_cache/corpus_tree.json
output tree : /Users/asharma/Desktop/Project Algorithm/tree_cache/corpus_tree_fixed.json


In [3]:
# ---- repair engine (tested; table section level = 2) ----
import os, re, json, difflib, datetime, uuid

# ---------- tree helpers (raw dicts; nothing dropped) ----------

def _norm(s):
    return re.sub(r"\s+", " ", (s or "").strip().lower())

def basename(p):
    return os.path.basename(str(p or "")).strip().lower()

def all_node_ids(tree):
    ids=set(); st=[tree]
    while st:
        n=st.pop(); ids.add(n.get("node_id"))
        st.extend(n.get("children",[]) or [])
    return ids

# find the document node for a given source file
def find_document_node(tree, filename):
    target=basename(filename); found=[None]
    def walk(n):
        if n.get("node_type")=="document":
            sf=(n.get("metadata",{}) or {}).get("source_file") or n.get("path","")
            nm=n.get("name","")
            if basename(sf)==target or basename(nm)==target:
                found[0]=n; return
        for c in n.get("children",[]) or []:
            walk(c)
            if found[0]: return
    walk(tree)
    return found[0]

# the level where the document's real sequence lives = the lowest-level node that has MORE THAN ONE child.
# returns (container_node, list_of_section_children). for the typical doc, that's the document node and its sections.
def section_container(doc_node):
    node=doc_node
    while True:
        kids=node.get("children",[]) or []
        if len(kids)==1 and kids[0].get("children"):     # single child that itself nests -> descend through it
            node=kids[0]; continue
        return node, kids

# every leaf chunk under a node, in reading order, with the SECTION it belongs to (the container's direct child)
def section_chunks_in_order(container, sections):
    out=[]
    for si,sec in enumerate(sections):
        def collect(n):
            for c in n.get("children",[]) or []:
                if c.get("node_type")=="chunk" or not c.get("children"):
                    out.append({"section_index":si,"section":sec,"chunk":c})
                else:
                    collect(c)
        if sec.get("node_type")=="chunk" or not sec.get("children"):
            out.append({"section_index":si,"section":sec,"chunk":sec})   # section is itself a leaf (rare)
        else:
            collect(sec)
    return out

# ---------- table detection / dedupe ----------

def serialize_table(rows):
    out=[]
    for r in rows or []:
        cells=[re.sub(r"\s+"," ",(c or "").strip()) for c in r]
        if any(cells): out.append(" | ".join(cells))
    return "\n".join(out)

def table_already_present(table_text, summary, doc_texts, doc_summaries, thresh=0.55):
    if _norm(summary) and _norm(summary) in doc_summaries: return True
    cells=[_norm(c) for c in re.split(r"[|\n]", table_text) if len(_norm(c))>=3]
    if not cells: return True
    for txt in doc_texts:
        t=_norm(txt)
        if sum(1 for c in cells if c in t)/len(cells) >= thresh: return True
    return False

def _sim(a,b):
    a=_norm(a)[:600]; b=_norm(b)[:600]
    if not a or not b: return 0.0
    return difflib.SequenceMatcher(None,a,b).ratio()

# choose the section-sibling index to insert AFTER: the section whose chunk text best matches the text
# directly PRECEDING the table (and, when possible, confirm the FOLLOWING text matches the next section)
def choose_section_position(chunk_rows, sections, preceding_text, following_text):
    if not chunk_rows:
        return len(sections)-1, "no chunks; appended at end"
    def best_section(text):
        if not text: return None,0.0
        scored=[(_sim(text, cr["chunk"].get("content") or cr["chunk"].get("summary","")), cr["section_index"])
                for cr in chunk_rows]
        s,i=max(scored); return (i,s) if s>0.12 else (None,s)
    prev_i,ps=best_section(preceding_text)
    next_i,ns=best_section(following_text)
    if prev_i is not None and next_i is not None and next_i==prev_i+1:
        return prev_i, "after the section preceding the table (next section confirmed)"
    if prev_i is not None:
        return prev_i, "after the section whose chunk directly precedes the table"
    if next_i is not None:
        return next_i-1, "before the section whose chunk directly follows the table"
    return len(sections)-1, "weak match; appended after last section"

# ---------- build the section node (one table chunk inside), matching the existing node format exactly ----------

def build_table_section(template_section, src_file, page, table_text, summary, all_ids, table_seq=0):
    tmpl_md=(template_section or {}).get("metadata",{}) or {}
    fname=os.path.basename(str(src_file or ""))
    # section label for a standalone table; mirrors the "<file> - <label>" naming of real sections
    label=f"Table {table_seq}" if table_seq else "Table"
    sec_name=f"{fname} - {label}"
    chunk_name=f"{fname} - {label} - table 0"
    sec_id=uuid.uuid4().hex
    while sec_id in all_ids: sec_id=uuid.uuid4().hex
    all_ids.add(sec_id)
    chunk_id=uuid.uuid4().hex
    while chunk_id in all_ids: chunk_id=uuid.uuid4().hex
    all_ids.add(chunk_id)
    src=tmpl_md.get("source_file") or src_file
    chunk={
        "node_id":chunk_id, "node_type":"chunk", "name":chunk_name, "path":"",
        "summary":summary, "content":table_text, "children":[],
        "metadata":{"source_file":src, "section":label, "unit_index":0,
                    "file_type":tmpl_md.get("file_type",".pdf"), "kind":"table", "page":page}
    }
    section={
        "node_id":sec_id, "node_type":"section", "name":sec_name, "path":"",
        "summary":summary, "content":"", "children":[chunk],
        "metadata":{"source_file":src, "section":label,
                    "level":2, "page":page}
    }
    return section

# ---------- top level: repair one PDF (mutates tree), with full-JSON above/below in the report ----------

def repair_one_pdf(tree, pdf_path, summarize=None, embed=None, make_embedding=False, verbose=True):
    fname=os.path.basename(pdf_path)
    rep={"file":fname,"tables_found":0,"already_present":0,"inserted":0,"insertions":[],"matched_in_tree":False}
    doc=find_document_node(tree, fname)
    if not doc:
        rep["warning"]="no document node found with this source_file/name; nothing inserted"; return rep
    rep["matched_in_tree"]=True
    container, sections = section_container(doc)
    try:
        blocks=extract_blocks(pdf_path)
    except Exception as e:
        rep["error"]=f"could not parse pdf: {type(e).__name__}: {e}"; return rep

    chunk_rows=section_chunks_in_order(container, sections)
    doc_texts=[cr["chunk"].get("content") or cr["chunk"].get("summary","") for cr in chunk_rows]
    doc_summaries={_norm(cr["chunk"].get("summary","")) for cr in chunk_rows}
    template_section=next((s for s in sections if s.get("node_type")=="section"), sections[0] if sections else doc)
    all_ids=all_node_ids(tree)
    table_seq=0
    for b in blocks:
        if b["type"]!="table": continue
        rep["tables_found"]+=1
        summary=summarize(b["text"], b.get("above","")+" "+b.get("below","")) if summarize else \
                "Table extracted from the document."
        if table_already_present(b["text"], summary, doc_texts, doc_summaries):
            rep["already_present"]+=1; continue
        pos,why=choose_section_position(chunk_rows, sections, b.get("above",""), b.get("below",""))
        src=(template_section.get("metadata",{}) or {}).get("source_file") or pdf_path
        new_section=build_table_section(template_section, src, b["page"], b["text"], summary, all_ids, table_seq=table_seq)
        table_seq+=1
        before_section = sections[pos] if 0<=pos<len(sections) else None
        after_section  = sections[pos+1] if pos+1<len(sections) else None
        container["children"].insert(pos+1, new_section)
        sections=container["children"]                       # refresh sibling list after insert
        # keep document.metadata.num_sections honest if it exists
        if isinstance(doc.get("metadata",{}),dict) and "num_sections" in doc["metadata"]:
            doc["metadata"]["num_sections"]=sum(1 for s in container["children"] if s.get("node_type")=="section")
        rep["inserted"]+=1
        rep["insertions"].append({
            "page":b["page"], "where":why,
            "SECTION_ABOVE": before_section, "INSERTED_SECTION": new_section, "SECTION_BELOW": after_section,
        })
        if verbose:
            print(f"\n{'#'*78}\nINSERT on page {b['page']} — {why}\n{'#'*78}")
            print("\n----- SECTION DIRECTLY ABOVE -----")
            print(json.dumps(before_section, indent=2, ensure_ascii=False) if before_section else "(start of document)")
            print("\n+++++ INSERTED SECTION (new) +++++")
            print(json.dumps(new_section, indent=2, ensure_ascii=False))
            print("\n----- SECTION DIRECTLY BELOW -----")
            print(json.dumps(after_section, indent=2, ensure_ascii=False) if after_section else "(end of document)")
        doc_texts.append(b["text"]); doc_summaries.add(_norm(summary))
    return rep


# ---------- PDF parsing: tables and the text immediately around them, in reading order ----------

def serialize_table(rows):
    out=[]
    for r in rows or []:
        cells=[re.sub(r"\s+"," ",(c or "").strip()) for c in r]
        if any(cells): out.append(" | ".join(cells))
    return "\n".join(out)

# returns ordered blocks: {"type":"table"/"text","page":n,"top":y,"text":...,"rows":...}
# each table also carries the text just above/below it on the page for anchoring
def extract_blocks(pdf_path):
    import pdfplumber
    blocks=[]
    with pdfplumber.open(pdf_path) as pdf:
        for pno,page in enumerate(pdf.pages, start=1):
            try: tables=page.find_tables()
            except Exception: tables=[]
            tboxes=[]
            for t in tables:
                rows=t.extract()
                txt=serialize_table(rows)
                if _norm(txt): tboxes.append((t.bbox, rows, txt))
            # page words for context text above/below each table
            try: words=page.extract_words(use_text_flow=True) or []
            except Exception: words=[]
            def text_between(y0,y1):
                ws=[w["text"] for w in words if y0 <= (w["top"]+w["bottom"])/2 <= y1]
                return re.sub(r"\s+"," "," ".join(ws)).strip()
            ph=page.height
            for bbox,rows,txt in tboxes:
                x0,top,x1,bottom=bbox
                above=text_between(max(0,top-260), top-2)[-500:]
                below=text_between(bottom+2, min(ph,bottom+260))[:500]
                blocks.append({"type":"table","page":pno,"top":top,"rows":rows,
                               "text":txt,"above":above,"below":below})
            # also keep a coarse page-text block (helps anchoring when a table has no nearby words)
            try: ptext=page.extract_text() or ""
            except Exception: ptext=""
            if _norm(ptext):
                blocks.append({"type":"text","page":pno,"top":0,"text":ptext})
    blocks.sort(key=lambda b:(b["page"], b["top"]))
    return blocks


# ---------- Ollama summarize / embed (same models + endpoint as the project) ----------

def make_ollama(url="http://localhost:11528", agent_model="gpt-oss:120b",
                embed_model="nomic-embed-text", keep_alive="30m", timeout=600):
    import ollama
    client=ollama.Client(host=url, timeout=timeout)
    def summarize(table_text, context=""):
        prompt=("Summarize what the following table contains and its purpose, in 2-4 sentences. "
                "Describe the actual data, columns, and topics. Do NOT use meta-language like 'this table'.\n\n"
                + (f"Surrounding context:\n{context[:600]}\n\n" if context else "")
                + f"TABLE:\n{table_text[:3000]}")
        try:
            r=client.chat(model=agent_model, messages=[{"role":"user","content":prompt}],
                          options={"temperature":0,"num_predict":400}, keep_alive=keep_alive, think=False)
            txt=(r["message"].get("content") or "").strip()
            if not txt: txt=(r["message"].get("thinking") or "").strip()
            return txt or "Tabular data extracted from the document."
        except TypeError:
            r=client.chat(model=agent_model, messages=[{"role":"user","content":prompt}],
                          options={"temperature":0,"num_predict":400}, keep_alive=keep_alive)
            return (r["message"].get("content") or "").strip() or "Tabular data extracted from the document."
    def embed(text):
        try:
            r=client.embeddings(model=embed_model, prompt=text or " ")
            return list(r["embedding"])
        except Exception:
            return None
    return summarize, embed



In [4]:
summarize, embed = make_ollama(OLLAMA_URL, AGENT_MODEL, EMBED_MODEL)
print('ollama ready:',AGENT_MODEL)

ollama ready: gpt-oss:120b


### Load (resumes from corpus_tree_fixed.json if it exists)

In [5]:
import json, shutil, datetime
# resume: if a partial output exists, keep building on it; otherwise start from the original tree
if FIXED_PATH.exists():
    with open(FIXED_PATH, encoding='utf-8') as f: tree=json.load(f)
    print('RESUMING from existing', FIXED_PATH.name)
else:
    with open(TREE_PATH, encoding='utf-8') as f: tree=json.load(f)
    print('starting fresh from', TREE_PATH.name)
# set of already-processed pdf paths
if PROGRESS_PATH.exists():
    with open(PROGRESS_PATH, encoding='utf-8') as f: processed=set(json.load(f))
else:
    processed=set()
def _count(n): return 1+sum(_count(c) for c in n.get('children',[]) or [])
print('tree has', _count(tree), 'nodes;', len(processed), 'files already processed')

RESUMING from existing corpus_tree_fixed.json
tree has 134632 nodes; 1371 files already processed


### Repair every PDF — checkpoints after each file

In [6]:
from tqdm.auto import tqdm
pdfs=sorted(set(FOLDERS_DIR.rglob('*.pdf'))|set(FOLDERS_DIR.rglob('*.PDF')))
todo=[p for p in pdfs if str(p) not in processed]
print(f'{len(pdfs)} pdfs total; {len(pdfs)-len(todo)} done, {len(todo)} to go')

def _save():
    tmp=FIXED_PATH.with_suffix('.tmp')
    with open(tmp,'w',encoding='utf-8') as f: json.dump(tree,f,ensure_ascii=False,indent=1)
    tmp.replace(FIXED_PATH)                       # atomic write so a crash never leaves a half-file
    ptmp=PROGRESS_PATH.with_suffix('.tmp')
    with open(ptmp,'w',encoding='utf-8') as f: json.dump(sorted(processed),f)
    ptmp.replace(PROGRESS_PATH)

reports=[]; F=I=D=0
bar=tqdm(todo, desc='repairing', unit='pdf')
for p in bar:
    try:
        r=repair_one_pdf(tree,str(p),summarize=summarize,embed=embed,verbose=False)
        reports.append(r); F+=r['tables_found']; I+=r['inserted']; D+=r['already_present']
    except Exception as e:
        print(f'  ! error on {p.name}: {type(e).__name__}: {e} (skipping; will retry next run)')
        continue
    processed.add(str(p))
    _save()                                       # checkpoint after EVERY file
    bar.set_postfix(inserted=I, present=D, file=p.name[:22])
print(f'\nDONE — found {F}, inserted {I}, already present {D}')
print('written to', FIXED_PATH)

/opt/homebrew/Cellar/jupyterlab/4.5.7_1/libexec/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2085 pdfs total; 1371 done, 714 to go


repairing: 100%|███████████████████████████████████████████████████| 714/714 [6:03:30<00:00, 30.55s/pdf, file=PD-015 CAPPT_0027_Ov_P, inserted=1192, present=2395]


DONE — found 3587, inserted 1192, already present 2395
written to /Users/asharma/Desktop/Project Algorithm/tree_cache/corpus_tree_fixed.json


### Back up the original tree

In [7]:
# a one-time backup of the ORIGINAL tree (the fixed file is already saved incrementally above)
stamp=datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
bk=TREE_PATH.with_name(f'corpus_tree.backup-{stamp}.json')
if not bk.exists(): shutil.copy2(TREE_PATH, bk)
print('original backed up ->', bk)

original backed up -> /Users/asharma/Desktop/Project Algorithm/tree_cache/corpus_tree.backup-20260627-180948.json


### Review anything unmatched

In [8]:
flag=[r for r in reports if r.get('warning') or r.get('error') or not r.get('matched_in_tree',True)]
print('files needing review this run:',len(flag))
for r in flag: print(' -',r['file'],'|',r.get('warning') or r.get('error') or 'no match')

files needing review this run: 15
 - Laboratory Cleaning Checklist 210225.pdf | no document node found with this source_file/name; nothing inserted
 - Laboratory Cleaning Checklist 210405.pdf | no document node found with this source_file/name; nothing inserted
 - Laboratory Cleaning Checklist 220228.pdf | no document node found with this source_file/name; nothing inserted
 - Laboratory Cleaning Checklist 220830.pdf | no document node found with this source_file/name; nothing inserted
 - Laboratory Cleaning Checklist 221104.pdf | no document node found with this source_file/name; nothing inserted
 - Laboratory Cleaning Checklist 221221.pdf | no document node found with this source_file/name; nothing inserted
 - D5000 Reagents 5067-5589_0006389838.pdf | no document node found with this source_file/name; nothing inserted
 - HS D1000 Reagents 5067-5585_0006421232.pdf | no document node found with this source_file/name; nothing inserted
 - HS D5000 Reagents 5067-5593_0006375521.pdf | no do

### Optional: reset to start over

In [9]:
# OPTIONAL — start completely over (deletes the partial output + progress, keeps the original tree)
# FIXED_PATH.unlink(missing_ok=True); PROGRESS_PATH.unlink(missing_ok=True); print('reset; next run starts fresh')